this file is my preliminary ridge linear model for predictive mapping to identify cell-type signatures that contribute to cell death
- handles collinearity well and incorporates penalization and maintains all coefficients (shrink not eliminate)
- conclusion because coefficients are so small, weak signal & now need to move to nonlinearity

In [1]:
# necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
# working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

# my dataset & then setting predictors and response variable
df = pd.read_csv("iso7spatialSig_means(2).csv")

signature_cols = df.columns[12:22]
X = df[signature_cols]
y = df["prop_dying"]
# print(X)

In [ ]:
# since the alpha was so small suggests ridge regression is more appropriate, validate
model = Pipeline([
    ("scaler", StandardScaler()),            
    ("enet", RidgeCV(
        alphas=[0.1, 1.0, 10.0],  
        cv=5               
    ))
])

In [ ]:
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('enet', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","[0.1, 1.0, ...]"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",None
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, te

In [ ]:
enet2 = model.named_steps["enet"]

coeffs = enet2.coef_
intercept = enet2.intercept_

for name, coef in zip(signature_cols, coeffs):
    print(f"{name}: {coef:.4f}")

In [ ]:
# is there any signal at all. because above very weak
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring="r2")
print(scores.mean())

In [ ]:
# answer was very little signal
# go back to logistic regression for binary response variable. 
from sklearn.linear_model import LogisticRegressionCV

y2 = df["exist_dying"]

In [ ]:
model2 = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegressionCV(
        penalty="l2",          # ridge
        cv=5,
        scoring="roc_auc",     # important
        max_iter=1000,
        n_jobs=-1
    ))
])

model2.fit(X, y2)

In [ ]:
logreg = model2.named_steps["logreg"]

coeffs = logreg.coef_[0]   
intercept = logreg.intercept_[0]

for name, coef in zip(signature_cols, coeffs):
    print(f"{name}: {coef:.4f}")

cd8_exhausted: -0.0139
cd4_effector: 0.0873
treg: 0.0609
nk: -0.0792
folr2_mac: 0.0656
monocytes: 0.4516
cdc1: 0.3364
cdc2: -0.4409
endothelial: 0.0431
fibroblast: -0.0962


In [ ]:
# barchart for visualization
coef_df = pd.Series(coeffs, index=signature_cols)

# sort for readability
coef_df = coef_df.sort_values()

plt.figure(figsize=(6, 4))
coef_df.plot(kind="barh")
plt.axvline(0)  # reference line at zero

plt.xlabel("Coefficient (standardized scale)")
plt.title("Elastic Net Coefficients")
plt.tight_layout()
plt.show()

In [ ]:
cross_val_score(model2, X, y2, cv=5, scoring="roc_auc") # measuring classifier ability to distinguish random prediction from actual
                                                        # 0.5 is random, above is better, below is worse
                                                        # unfortunately, this model returns ~0.5